# Custom Contact Definitions

In [5]:
import numpy as np
from openbabel import openbabel as ob

from lahuta import Luni, NeighborPairs
from lahuta.contacts import ContactAnalysis
from lahuta.tests.data import X2

In [6]:
luni = Luni(str(X2(pdb=False)))
ns = luni.compute_neighbors(res_dif=1, radius=5.0)

To define new contact analysis functions, we can, of course, simply operate on the `ns` object created above.  

We can also use the `ContactAnalysis` method which makes things simpler and more consistent. 

In this Notebook, we will inherit from the `ContactAnalysis` class to define new classes that will contain our computation logic. One thing to know is that when we inherit from `ContactAnalysis` we need to define one of two methods: `compute` or `compute_elementwise`. The latter is easier to use but it involves iterating over all pairs of neighboring atoms in the systm, which can be slow for large systems. The former is more efficient as it uses vectorized operations, but it requires a bit more knowledge of NumPy and array operations.

# Salt Bridges

Salt bridges are formed between oppositely charged residues. 

Here we will define the same class twice using the two methods mentioned above. You'll notice that the `compute_elementwise` is more expressive and easier to read, but it is also slower. In contrast, the `compute` method is significantly faster, but it requires knowledge of the NeighborPairs object along with NumPy array operations to be able to use it.

#### Elementwise

In [7]:
sel_acidic = "(resname ASP GLU) and (name OE* OD*)"
sel_basic = "(resname ARG LYS) and (name NH* NZ)"

class SaltBridgeContacts(ContactAnalysis):
    """Salt bridge contacts."""
    SALT_BRIDGE_CUTOFF_DISTANCE = 4.0

    ACIDICT_INDICES = luni.to("mda").select_atoms(f'({sel_acidic})').indices
    BASIC_INDICES = luni.to("mda").select_atoms(f'{sel_basic}').indices
    def compute_elementwise(self, atom1, atom2, distance) -> NeighborPairs | None:
        """Compute salt bridge contacts.

        This method is called automatically upon initialization of the class with the listed arguments.
        It is called for each neighboring pair of atoms in the system.

        Args:
            atom1 (Atom): First atom.
            atom2 (Atom): Second atom.
            distance (float): Distance between the two atoms.

        Returns:
            NeighborPairs | None: Salt bridge contacts.
        """
        if distance >= self.SALT_BRIDGE_CUTOFF_DISTANCE: return None

        if (
            (atom1.index in self.ACIDICT_INDICES and atom2.index in self.BASIC_INDICES) or
            (atom1.index in self.BASIC_INDICES and atom2.index in self.ACIDICT_INDICES)
        ):
            return ns.new(
                np.array([atom1.index, atom2.index], dtype=np.int32).reshape(1, 2), 
                distances=np.array([distance])
            )

In [8]:
sb = SaltBridgeContacts(ns)
sb.results

<Lahuta NeighborPairs class containing 7 atoms and 4 pairs>

#### Vectorized

In [9]:
sel_acidic = "(resname ASP GLU) and (name OE* OD*)"
sel_basic = "(resname ARG LYS) and (name NH* NZ)"

class SaltBridgeContacts(ContactAnalysis):
    """Salt bridge contacts."""
    SB_DISTANCE = 4.0
    ACIDIC = luni.to("mda").select_atoms(f'({sel_acidic})').indices
    BASIC = luni.to("mda").select_atoms(f'{sel_basic}').indices
    def compute(self) -> NeighborPairs | None:
        """Compute salt bridge contacts.

        This method is called automatically upon initialization of the class. Within this method, you have access to the
        NeighborPairs object that was passed to the class upon initialization via the self.ns attribute. 
        It is called once for all neighboring pairs of atoms in the system.

        Returns:
            NeighborPairs | None: Salt bridge contacts.
        """
        ns12 = self.ns.index_filter(self.ACIDIC, partner=1).index_filter(self.BASIC, partner=2).distance_filter(self.SB_DISTANCE)
        ns21 = self.ns.index_filter(self.BASIC, partner=1).index_filter(self.ACIDIC, partner=2).distance_filter(self.SB_DISTANCE)

        return ns12 + ns21

In [10]:
sb = SaltBridgeContacts(ns)
sb.results

<Lahuta NeighborPairs class containing 7 atoms and 4 pairs>

# Van der Waals Contacts

The definition of VdW contacts depends only on the distance between two atoms as a function of their van der Waals radii.

Again, to highlight the difference between the two methods, we will define two classes that produce the same result. The first one will use the `compute_elementwise` method and the second one will use the `compute` method.

#### Elementwise

In [11]:
EPSILON = 0.5
class VdWContacts(ContactAnalysis):
    """Van der Waals contacts between protein atoms."""
    PROTEIN_INDICES = luni.to("mda").select_atoms('protein').indices
    def compute_elementwise(self, atom1, atom2, distance) -> NeighborPairs | None:
        """Compute the VdW contacts between two atoms."""
        if (
            atom1.index in self.PROTEIN_INDICES and
            atom2.index in self.PROTEIN_INDICES and
            distance < (EPSILON + atom1.vdw_radii + atom2.vdw_radii)
        ):
            if atom1.resname == atom2.resname == "CYS":
                return None
            
        
            return NeighborPairs(luni).new(
                np.array([[atom1.index, atom2.index]], dtype=np.int32), 
                distances=np.array([distance])
            )

Initialize the class and access the results. Note that for a PDB ID file such as 1GZM this takes roughly 2 seconds.

In [12]:
vdw = VdWContacts(ns)
vdw.results

<Lahuta NeighborPairs class containing 382 atoms and 474 pairs>

#### Vectorized

In [13]:
EPSILON = 0.5
class VdWContacts(ContactAnalysis): 
    """Van der Waals contacts between protein atoms."""
    PROTEIN_INDICES = luni.to("mda").select_atoms('protein').indices
    def compute(self) -> NeighborPairs | None:
        """Compute the VdW contacts between neighboring atoms."""
        ns = self.ns.index_filter(self.PROTEIN_INDICES, partner=1).index_filter(self.PROTEIN_INDICES, partner=2)

        radii_mask = ns.distances < luni.to("mda")[ns.pairs].vdw_radii.sum(axis=1) + EPSILON
        cys_mask = np.where(~np.all(ns.resnames == 'CYS', axis=1), True, False)
        mask = radii_mask & cys_mask

        return ns.new(ns.pairs[mask], distances=ns.distances[mask])

For the same PDB ID file, the `compute` method takes only 2 ms. 

In [14]:
vdw = VdWContacts(ns)
vdw.results

<Lahuta NeighborPairs class containing 382 atoms and 474 pairs>

# Hydrophobic Contacts

Hydrophobic contacts have a slighlty more complicated definition. 

In [15]:
from MDAnalysis.lib.NeighborSearch import AtomNeighborSearch

EPSILON = 0.5
class HydrophobicContacts(ContactAnalysis): 
    """Hydrophobic contacts between protein atoms."""

    aa_sel = "(resname ALA PHE GLY ILE LEU PRO VAL TRP) and (element C)"
    VALID_INDICES = luni.to("mda").select_atoms(aa_sel).indices

    def compute_elementwise(self, atom1, atom2, distance) -> NeighborPairs | None: 
        """Compute the hydrophobic contacts between two atoms."""
        if (
            atom1.index in self.VALID_INDICES and
            atom2.index in self.VALID_INDICES and
            self.has_valid_neighbors_mol(luni.to("mol"), atom1.index, {6, 1}) and # when using OB
            self.has_valid_neighbors_mol(luni.to("mol"), atom2.index, {6, 1}) and # when using OB
            # self.has_valid_neighbors_mda(luni.to("mda"), atom2, 1.7, ['C', 'H']) and # if using MDA
            # self.has_valid_neighbors_mda(luni.to("mda"), atom1, 1.7, ['C', 'H']) and # if using MDA
            distance < EPSILON + 2 * 1.7 and
            distance < (EPSILON + atom1.vdw_radii + atom2.vdw_radii)
        ):
            if atom1.resname == atom2.resname == "CYS":
                return None
            return ns.new(
                np.array([[atom1.index, atom2.index]], dtype=np.int32), 
                distances=np.array([distance])
            )

    @staticmethod
    def has_valid_neighbors_mol(mol, atom_id: int, valid_neighbors: set[int]) -> bool:
        """Check if an atom has valid neighbors using bond information (valid only for small cutoffs)."""
        ob_atom = mol.GetAtomById(int(atom_id))
        for bond in ob.OBAtomBondIter(ob_atom):
            b_atom = bond.GetBeginAtom()
            other_atom = bond.GetEndAtom() if b_atom == ob_atom else b_atom
            if other_atom.GetAtomicNum() not in valid_neighbors:
                return False

        return True

    @staticmethod
    def has_valid_neighbors_mda(atom_group, atom, radius: float, valid_neighbors: list[str]):
        """Check if an atom has valid neighbors using a NeighborSearch object. This is valid 
        for both small and large cutoffs, but is significantly slower.
        """
        result = AtomNeighborSearch(atom_group).search(atoms=atom, radius=radius)
        return len(result) > 0 and np.isin(result.elements, valid_neighbors).all()

In [16]:
# %%timeit
hc = HydrophobicContacts(ns) 
hc.results

<Lahuta NeighborPairs class containing 9 atoms and 5 pairs>

The above code will produce only `hp` contacts without consideration of ligands. It is, however, trivial to add ligand information to the `hp` contacts.

We just need to get the selection of indices that belong to ligands and use that to filter the pairs of neighboring atoms. One thing to note is that the above `VALID_INDICES` selection will be mutually exclusive with the ligand selection below. So you need to handle the different cases in your code.

In [17]:
# NOTE that solvent and lipid resnames are incomplete
solv_resnames = "H2O HH0 OHH HOH OH2 SOL WAT TIP TIP2 TIP3 TIP4 T3P IP3"
lipid_resnames = "DLPE DMPC DPPC GPC LPPC PALM PC PGCL POPC POPE"

# everything that is not a protein, nucleic acid, solvent, or lipid is considered a ligand
lig_sel = f"(not resname {solv_resnames} and not resname {lipid_resnames} and not protein and not nucleic) and (element C)"
LIGAND_INDICES = luni.to("mda").select_atoms(lig_sel).indices
print (LIGAND_INDICES.size)

34


# Pi and T-stacking contacts

These contacts are more challenging since they require the definition of a plane. We are computing and comparing distances and angles between two planes.

I've decided to write these contact classes based on available utilities in lahuta. As such, they are not exactly the same. Specifically, rather than hard-code the residue names we support, the code below uses the `lahuta` package to identify aromaticity information from the system. This way, we automatically get both the center and normal vector of the aromatic ring. 

This allows us to extend the definition of pi-stacking contacts to include any aromatic residue (even ligands), which are not supported by the original definition in getcontacts. 

In [18]:
from lahuta.utils.math import calc_vec_angle, calc_vec_angle
from lahuta.utils.ob import enumerate_rings
from MDAnalysis.lib import distances as mda_distances

STACK_CUTOFF_ANGLE = 30.0
STACK_PSI_ANGLE = 45.0

class StackingContacts(ContactAnalysis):
    """Stacking contacts between aromatic rings."""

    SOFT_CUTOFF = {'ts': 6.0, 'ps': 10.0}
    DISTANCE_CUTOFF = {'ts': 5.0, 'ps': 7.0}

    def __init__(self, ns, itype='ps'):
        self.itype = itype
        self.rings = enumerate_rings(luni.to("mol"))
        super().__init__(ns)

    def compute(self) -> NeighborPairs:
        """Compute stacking contacts between aromatic rings."""
        pairs, distances = mda_distances.self_capped_distance(
            self.rings.centers,
            max_cutoff=self.SOFT_CUTOFF[self.itype],
            min_cutoff=2.5,
            method='nsgrid'
        )

        # filter distances
        distances_filter = distances < self.DISTANCE_CUTOFF[self.itype]
        pairs, distances = pairs[distances_filter], distances[distances_filter]

        # filter angles and psi angles
        angles = calc_vec_angle(self.rings.normals[pairs[:, 0]], self.rings.normals[pairs[:, 1]])
        if self.itype == 'ts':
            angles = np.fabs(angles - 90)

        psi_angles = calc_vec_angle(
            self.rings.centers[pairs[:, 0]] - self.rings.centers[pairs[:, 1]], 
            self.rings.normals[pairs[:, 0]]
        )

        angles_filter = (
            (angles < STACK_CUTOFF_ANGLE) & 
            ~((psi_angles > STACK_PSI_ANGLE) & (psi_angles < 180 - STACK_PSI_ANGLE))
        )

        # save distances between aromatic centers
        pairs = pairs[angles_filter]
        self.center_distances = distances[angles_filter]

        return NeighborPairs(luni).new(self.rings.first_atom_idx[pairs], distances=self.center_distances)


In [19]:
sc = StackingContacts(ns, itype='ts')
sc.results.labels

array([[('C1B', '90', 'HEC'), ('ND1', '15', 'HIS')]],
      dtype=[('names', '<U25'), ('resids', '<U25'), ('resnames', '<U25')])

# Pi-Cation contacts

We compute and compare distances and angles between a plane and a point.

Again, same as above, we use the `lahuta` package to identify aromaticity information from the system. This way, we automatically get both the center and normal vector of the aromatic rings. We support any type of aromatic residue (even ligands), which are not supported by the original definition in getcontacts. 

Note that this definion is not meant to be an exact replica of the original getcontacts definition. Rather, it is meant to highlight the flexibility of `lahuta` and how it can be used to define new contact types. To show why this is the better approach, I've re-written Pi-Cation contacts using two different ways: 
- The `lahuta` way, which uses existing capabilities of the library
- An exact re-write of the original getcontacts definition which should produce the same results (see further below). You'll notice that this is much more verbose and less flexible.

#### Easy to use and understand 

In [20]:
from lahuta.utils.array_utils import unique_indices

PI_CATION_CUTOFF_DISTANCE  = 6.0 
PI_CATION_CUTOFF_ANGLE = 60.0

basic_arg_lys = "(resname ARG LYS) and (name NH* NZ)"
basic_his = "((resname HIS HSD HSE HSP HIE HIP HID) and (name ND1 NE2))"

class PiCationNeighbors(ContactAnalysis):
    """Pi-cation contacts between aromatic rings and basic residues."""
    BASIC_INDICES = luni.to("mda").select_atoms(f'protein and ({basic_arg_lys} or {basic_his})')

    def __init__(self, ns):
        self.rings = enumerate_rings(luni.to("mol"))
        super().__init__(ns)

    def compute(self):
        """Compute pi-cation contacts between aromatic rings and basic residues."""
        pairs, distances = mda_distances.capped_distance(
            self.rings.centers, 
            self.BASIC_INDICES.positions, 
            max_cutoff=PI_CATION_CUTOFF_DISTANCE
        )

        cation_arom_centroid_vec = self.BASIC_INDICES.positions[pairs[:, 1]] - self.rings.centers[pairs[:, 0]]
        angles = calc_vec_angle(self.rings.normals[pairs[:, 0]], cation_arom_centroid_vec)
        angles_filter = angles <= PI_CATION_CUTOFF_ANGLE
        
        pairs, distances = pairs[angles_filter], distances[angles_filter]

        arr = np.zeros((len(pairs), 2), dtype=np.int32)
        arr[:, 0] = self.BASIC_INDICES[pairs[:, 1]].indices
        arr[:, 1] = self.rings.first_atom_idx[pairs[:, 0]]

        unique_resid_indices = unique_indices(luni.to("mda")[arr].resids)
        arr = arr[unique_resid_indices]
        distances = distances[unique_resid_indices]

        return ns.new(luni.to("mda")[arr].indices, distances=distances)


#### Re-write such that it produces the same results

Notice how much more involved the solution is. 

In [21]:
from lahuta.utils.math import distance, calc_vec_angle

PI_CATION_CUTOFF_DISTANCE  = 6.0 
PI_CATION_CUTOFF_ANGLE = 60.0

sel_basic = "(resname ARG LYS) and (name NH* NZ)"
basic_his = "((resname HIS HSD HSE HSP HIE HIP HID) and (name ND1 NE2))"

resnames = {
    "PHE": ["CG", "CE1", "CE2"],
    "TRP": ["CD2", "CZ2", "CZ3"],
    "TYR": ["CG", "CE1", "CE2"],
    "HIS HSD HSE HSP HIE HIP HID": ["CG", "CE1", "CD2"]
}

aromatic_selections = [f"((resname {res}) and (name {' '.join(names)}))" for res, names in resnames.items()]

class PiCationNeighbors(ContactAnalysis):
    """Pi-cation contacts between aromatic rings and basic residues."""
    AROMAIC_INDICES = luni.to("mda").select_atoms(f'protein and ({" or ".join(aromatic_selections)})')
    BASIC_INDICES = luni.to("mda").select_atoms(f'protein and ({sel_basic} or {basic_his})')

    def __init__(self, ns, max_cutoff=10.0):
        self.max_cutoff = max_cutoff

        pairs, distances = self.compute_neighbors(max_cutoff=max_cutoff)
        ns = ns.new(pairs, distances=distances)
        self.ns = ns
        self.mol = luni.to("mol")
        super().__init__(ns)

    def compute_elementwise(self, atom1, atom2, distance) -> NeighborPairs | None:
        """Compute pi-cation contacts between aromatic rings and basic residues."""
        if (
            (atom1.index in self.AROMAIC_INDICES.indices and atom2.index in self.BASIC_INDICES.indices) or
            (atom1.index in self.BASIC_INDICES.indices and atom2.index in self.AROMAIC_INDICES.indices)
        ):
            if self.mol.GetBond(self.mol.GetAtomById(int(atom1.index)), self.mol.GetAtomById(int(atom2.index))):
                return None
            
            return ns.new(np.array([atom1.index, atom2.index], dtype=np.int32).reshape(1, 2), distances=np.array([distance]))

    def compute_neighbors(self, max_cutoff=10.0):
        """Compute pi-cation contacts between aromatic rings and basic residues."""
        pairs, distances = mda_distances.capped_distance(
            self.AROMAIC_INDICES.positions, 
            self.BASIC_INDICES.positions, 
            max_cutoff=max_cutoff
        )

        arr = np.zeros((len(pairs), 2), dtype=np.int32)
        arr[:, 0] = self.BASIC_INDICES[pairs[:, 1]].indices
        arr[:, 1] = self.AROMAIC_INDICES[pairs[:, 0]].indices

        return luni.to("mda")[arr].indices, distances

class PiCationContactsExact:
    """Pi-cation contacts between aromatic rings and basic residues."""
    def __init__(self, ns):
        pcns = PiCationNeighbors(ns)
        self.ns = pcns.results
        self.rings = enumerate_rings(luni.to("mol"))
        self.results = self.compute()

    def _cation_aromatic_pairs(self):
        arr = self._prepare_neighbor_array()
        idx_eq_3 = self._get_idx_eq_3(arr)
        res = self._get_res(idx_eq_3)
        return res

    def _prepare_neighbor_array(self):
        col1 = self.ns.indices[:, 0]
        col2 = self.ns.chainIDs[:, 1].astype(str) + self.ns.resnames[:, 1] + self.ns.resids[:, 1].astype(str)
        return col1.astype(str) + col2

    def _get_idx_eq_3(self, arr):
        _, idx, counts = np.unique(arr, return_index=True, return_counts=True)
        idx_eq_3 = idx[counts == 3]
        return np.isin(arr, arr[idx_eq_3])
    
    def _get_res(self, idx_eq_3):
        indices = self.ns.indices[idx_eq_3]
        ix = np.lexsort((indices[:, 1], indices[:, 0]))
        return indices[ix]
            
    def _coords_from_chunks(self, chunks):
        cation_coords = luni.to("mda")[chunks[:, 0, 0]].positions
        arom_atom_coords = luni.to("mda")[chunks[:, :, 1]].positions
        return cation_coords, arom_atom_coords

    def _compute_vector_normals(self, arom_atom_coords):
        v1 = arom_atom_coords[:, 2] - arom_atom_coords[:, 0]
        v2 = arom_atom_coords[:, 1] - arom_atom_coords[:, 0]
        aromatic_plane_norm_vecs = np.cross(v1, v2)
        return aromatic_plane_norm_vecs

    def _filter_pication_interactions(self, cation_to_centroid_distances, cation_norm_offset_angles):
        interactions = np.logical_and(
            cation_to_centroid_distances <= PI_CATION_CUTOFF_DISTANCE,
            cation_norm_offset_angles <= PI_CATION_CUTOFF_ANGLE
        )
        return interactions

    def compute(self):
        """Compute pi-cation contacts between aromatic rings and basic residues."""
        cation_aromatic_pairs = self._cation_aromatic_pairs()
        chunks = cation_aromatic_pairs.reshape(-1, 3, 2)
        
        cation_coords, arom_atom_coords = self._coords_from_chunks(chunks)
        aromatic_centroids = arom_atom_coords.mean(axis=1)
        cation_centroid_distances = distance(cation_coords, aromatic_centroids)

        aromatic_plane_norm_vecs = self._compute_vector_normals(arom_atom_coords)
        cation_arom_centroid_vec = cation_coords - aromatic_centroids
        cation_norm_offset_angles = calc_vec_angle(aromatic_plane_norm_vecs, cation_arom_centroid_vec)

        interactions = self._filter_pication_interactions(cation_centroid_distances, cation_norm_offset_angles)

        pairs = chunks[interactions][:, 0, :]
        pairs_coords = luni.to("mda")[pairs].positions
        distances = distance(pairs_coords[:, 0], pairs_coords[:, 1])

        return self.ns.new(pairs, distances=distances)

In [22]:
pc_exact = PiCationContactsExact(ns)
pc_exact.results.labels

array([[('NZ', '78', 'LYS'), ('CD2', '31', 'TRP')]],
      dtype=[('names', '<U25'), ('resids', '<U25'), ('resnames', '<U25')])